# 02 — DataFrames and Spark SQL express the same work

**Student exercise**

Complete the tasks in order. Each task includes an expected result, three hints, and a check. All sample data and setup code are included in this notebook.

## 1. Spark configuration

Start a fresh kernel. Use Spark 3.5.x / Scala 2.12 and Java 17. Set `STUDENT_ID` to a lowercase name containing letters, numbers or underscores. Set `SPARK_MASTER=local[2]` for a local Spark session.

In [ ]:
import os
import socket
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

master_url = os.environ.get("SPARK_MASTER", f"spark://{socket.gethostname()}:7077")
STUDENT_ID = "student01"
assert STUDENT_ID and STUDENT_ID[0].islower() and all(c in "abcdefghijklmnopqrstuvwxyz0123456789_" for c in STUDENT_ID)

spark = (
    SparkSession.builder
    .appName("Exercise-02")
    .master(master_url)
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.redaction.regex", "(?i)secret|password|token|access[.]?key|credential")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## 2. Sample data

Run this cell, inspect the records, and use the supplied variables in the tasks.

In [ ]:
orders = spark.createDataFrame([
    (101,1,40.0,'2026-09-01'), (102,2,50.0,'2026-09-01'),
    (103,1,30.0,'2026-09-02'), (104,3,20.0,'2026-09-02')],
    'id INT, customer_id INT, amount DOUBLE, order_date STRING')
orders.show()

## 3. Result checks

This small function checks output values and duplicate counts. It is included here so the notebook can run on its own.

In [ ]:
def check_frame(df, columns, expected):
    assert df is not None, "Complete the task before running the check"
    assert set(columns).issubset(df.columns), "Missing output columns"
    def normalize(row):
        return tuple(round(value, 6) if isinstance(value, float) else value for value in row)
    actual = sorted(repr(normalize(row)) for row in df.select(*columns).collect())
    wanted = sorted(repr(normalize(row)) for row in expected)
    assert actual == wanted, f"Expected {wanted}, got {actual}"
    print("PASS:", columns)

## Task 1: Filter and project with DataFrames

Select id/customer_id/amount for orders with amount>=35; sort descending by amount.

**Expected result:** Orders 101 and 102.

In [ ]:
df_result = None  # TODO: filter, select, orderBy

**Hint 1:** Filtering does not run a Spark job until an action occurs.

**Hint 2:** Use filter(F.col("amount") >= 35); sort using F.desc.

**Hint 3:** orders.filter(...).select(...).orderBy(...).

### Check

In [ ]:
check_frame(df_result,['id','customer_id','amount'],[(101,1,40.0),(102,2,50.0)])

## Task 2: Write equivalent SQL

Register orders_input as a temporary view and produce sql_result with the same three columns and filter.

**Expected result:** SQL and DataFrame results agree.

In [ ]:
sql_result = None  # TODO: createOrReplaceTempView, spark.sql

**Hint 1:** A temporary view names a DataFrame plan; it does not create a persistent catalog table.

**Hint 2:** createOrReplaceTempView is called on a DataFrame, spark.sql is called on the session.

**Hint 3:** SELECT ... FROM orders_input WHERE ... ORDER BY amount DESC.

### Check

In [ ]:
check_frame(sql_result,['id','customer_id','amount'],[(101,1,40.0),(102,2,50.0)])

## Task 3: Calculate a derived SQL column

For all orders calculate amount_with_tax=amount*1.10; retain id. Verify no base data changes.

**Expected result:** Four taxed amounts: 44,55,33,22.

In [ ]:
taxed = None  # TODO: SELECT an expression AS amount_with_tax

**Hint 1:** Projection computes a new value; it does not update orders_input.

**Hint 2:** Alias the expression so the expected column name is present.

**Hint 3:** SELECT id, amount * 1.10 AS ... FROM orders_input.

### Check

In [ ]:
check_frame(taxed,['id','amount_with_tax'],[(101,44.0),(102,55.0),(103,33.0),(104,22.0)])
assert orders.count()==4

## More practice

Use explain() on both filter plans. Register another view and join it using SQL.

## Common mistakes

A Python variable is not automatically a SQL relation. Avoid confusing an alias with a stored column.

## Finish

Stop Spark before starting another notebook. For write exercises, run setup again before repeating append tasks. S3 data remains available after stopping Spark.

In [ ]:
spark.stop()